<a href="https://colab.research.google.com/github/RedGummyBear/ImmunomodulatorWerk/blob/main/OffTarget_Screening_V5vsV9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q rdkit

In [ ]:
# Cell 2
import pandas as pd, numpy as np, json, math, warnings, io
warnings.filterwarnings('ignore')
from rdkit import Chem
from rdkit.Chem import Descriptors, Crippen, rdMolDescriptors

In [ ]:
def potency_nM(smiles):                # placeholder
    return 500.0

def deltaG(Kd_nM):                     # kcal mol-1, 298 K
    return 1.987e-3 * 298 * math.log(Kd_nM * 1e-9)

def off_target_alert(smiles):          # simple rule-based
    mol = Chem.MolFromSmiles(smiles)
    if not mol: return 1.0
    # alert if >2 aryl rings + clogP > 4
    return float(Descriptors.NumAromaticRings(mol) > 2 and Crippen.MolLogP(mol) > 4)

In [ ]:
def evaluate(list_smiles, ref_smiles=None, ref_Kd_nM=500):
    records = []
    for smi in list_smiles:
        mol = Chem.MolFromSmiles(smi)
        if not mol: continue
        ic50 = potency_nM(smi)
        d = {'smiles': smi,
             'IC50_nM': ic50,
             'ΔG_kcal': deltaG(ic50),
             'off_target_p': off_target_alert(smi)}
        if ref_smiles and Chem.MolFromSmiles(ref_smiles):
            ref_ic50 = potency_nM(ref_smiles)
            d['ΔΔG_kcal'] = deltaG(ic50) - deltaG(ref_ic50)
        else:
            d['ΔΔG_kcal'] = 0.0
        records.append(d)
    return pd.DataFrame(records)

In [ ]:
candidates = [
    'COc1ccc(CO)c(F)c1-c1nc2ccnc2c1C(=O)Nc1ccc(OCCS(=O)(=O)CCO)cc1',           # V5
    'COc1ccc(CO)c(F)c1-c1nc2ccnc2c1C(=O)Nc1ccc(OCC(C)N2CCOCC2)cc1'              # V9
]

df = evaluate(candidates,
              ref_smiles='COc1ccc(CO)c(F)c1-c1nc2ccnc2c1C(=O)Nc1ccc(OCCS(=O)(=O)CCO)cc1',
              ref_Kd_nM=500)

print(df.round(2))

                                              smiles  IC50_nM  ΔG_kcal  \
0  COc1ccc(CO)c(F)c1-c1nc2ccnc2c1C(=O)Nc1ccc(OCCS...    500.0    -8.59   
1  COc1ccc(CO)c(F)c1-c1nc2ccnc2c1C(=O)Nc1ccc(OCC(...    500.0    -8.59   

   off_target_p  ΔΔG_kcal  
0           0.0       0.0  
1           0.0       0.0  
